# VN30 / CatBoost

**Historical exploratory experiment — not a validated benchmark.**

Source workspace filename: `VN30_from_JSON copy 2.ipynb`. See `../README.md` and `../../docs/source_manifest.json` for status and provenance.

Saved outputs were cleared for publication. Dataset paths were made relative; run from `notebooks/exploratory/`. Model dependencies and data access must be configured separately. Do not quote accuracy from this notebook without reproducing its split, target alignment, and calibration protocol.


In [ ]:
############# VN30 Lowest Price Forecasting với CatBoost

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta, datetime
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["PYTHONWARNINGS"] = "ignore"

# CatBoost imports
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

print("[INFO] CatBoost Forecasting Environment Ready")

In [ ]:
# ========== ĐỌC DỮ LIỆU VN30 TỪ FILE CSV ==========
def load_vn30_data(filepath: str) -> pd.DataFrame:
    """
    Đọc dữ liệu VN30 từ file CSV, lấy cột Thấp (giá thấp nhất trong ngày) và KL (Volume).
    Trả về df ['ds','y','volume'].
    
    File format:
    - Ngày: định dạng dd/mm/yyyy
    - Thấp: giá thấp nhất (có dấu phẩy ngăn cách hàng nghìn)
    - KL: khối lượng giao dịch (VD: 416.43M, 520.49M, 1.15B)
    """
    try:
        print(f"[INFO] Đang đọc dữ liệu từ {filepath}...")
        
        # Đọc CSV
        df = pd.read_csv(filepath, encoding='utf-8')
        
        # Kiểm tra columns
        required_cols = ['Ngày', 'Thấp', 'KL']
        for col in required_cols:
            if col not in df.columns:
                raise ValueError(f"Không tìm thấy cột '{col}' trong file CSV. Columns: {df.columns.tolist()}")
        
        # Đổi tên cột
        df = df.rename(columns={'Ngày': 'ds', 'Thấp': 'y', 'KL': 'volume'})
        
        # Convert ngày từ dd/mm/yyyy sang datetime
        df['ds'] = pd.to_datetime(df['ds'], format='%d/%m/%Y', errors='coerce')
        
        # Xử lý cột y: loại bỏ dấu phẩy và chuyển sang số
        # VD: "2,001.38" -> 2001.38
        if df['y'].dtype == 'object':
            df['y'] = df['y'].str.replace(',', '').str.replace('"', '')
        df['y'] = pd.to_numeric(df['y'], errors='coerce')
        
        # Xử lý cột volume: convert từ "416.43M" sang số
        # VD: "1.15B" -> 1150000000, "416.43M" -> 416430000, "1.5K" -> 1500
        if df['volume'].dtype == 'object':
            def parse_volume(val):
                if pd.isna(val) or val == '':
                    return np.nan
                val = str(val).strip()
                if 'B' in val:
                    return float(val.replace('B', '')) * 1_000_000_000
                elif 'M' in val:
                    return float(val.replace('M', '')) * 1_000_000
                elif 'K' in val:
                    return float(val.replace('K', '')) * 1_000
                else:
                    try:
                        return float(val)
                    except:
                        return np.nan
            df['volume'] = df['volume'].apply(parse_volume)
        
        df['volume'] = pd.to_numeric(df['volume'], errors='coerce')
        
        # Clean data
        df = df.dropna(subset=['ds', 'y', 'volume']).sort_values('ds')
        df = df[df['y'] > 0]
        
        # Keep only ds, y, volume columns
        df = df[['ds', 'y', 'volume']].reset_index(drop=True)
        
        print(f"[INFO] ✅ Đọc thành công {len(df)} dòng dữ liệu từ file CSV.")
        print(f"[INFO] Khoảng thời gian: {df['ds'].min().date()} → {df['ds'].max().date()}")
        return df
        
    except Exception as e:
        print(f"[ERROR] Lỗi khi đọc file: {e}")
        raise

# Đường dẫn file CSV
CSV_FILE = "../../data/raw/vn30.csv"

df = load_vn30_data(CSV_FILE)
print(f"\n[INFO] Nguồn dữ liệu: VN30 Historical Data (Lowest price + Volume) | Số dòng: {len(df)}")
print(f"[INFO] Thống kê giá: Min={df['y'].min():.2f}, Max={df['y'].max():.2f}, Mean={df['y'].mean():.2f}")
print(f"[INFO] Thống kê Volume: Min={df['volume'].min():.0f}, Max={df['volume'].max():.0f}, Mean={df['volume'].mean():.0f}")
df.head(10)

In [ ]:
print(len(df))

In [ ]:
df.dropna()

In [ ]:
df.fillna(method='ffill', inplace=True)

In [ ]:
# ========== TRAIN CATBOOST MODEL WITH VOLUME FEATURES ==========
def create_lagged_features(df, lags=150):
    """
    Tạo features từ giá trị quá khứ (lagged features) + volume để CatBoost sử dụng
    """
    df_features = df.copy()
    
    # Lagged price features
    for lag in range(1, lags + 1):
        df_features[f'lag_{lag}'] = df_features['y'].shift(lag)
    
    # Lagged volume features
    for lag in range(1, 31):  # 30 ngày volume lags
        df_features[f'volume_lag_{lag}'] = df_features['volume'].shift(lag)
    
    # Tính thêm các thống kê từ cửa sổ quá khứ - price
    df_features['ma7'] = df_features['y'].rolling(window=7).mean()
    df_features['ma30'] = df_features['y'].rolling(window=30).mean()
    df_features['volatility_7'] = df_features['y'].rolling(window=7).std()
    df_features['volatility_30'] = df_features['y'].rolling(window=30).std()
    df_features['roc_7'] = df_features['y'].pct_change(7)
    df_features['roc_30'] = df_features['y'].pct_change(30)
    
    # Volume features
    df_features['volume_ma7'] = df_features['volume'].rolling(window=7).mean()
    df_features['volume_ma30'] = df_features['volume'].rolling(window=30).mean()
    df_features['volume_ratio_7'] = df_features['volume'] / (df_features['volume_ma7'] + 1e-8)  # Volume spike detector
    df_features['volume_trend'] = df_features['volume'].rolling(window=7).sum()
    
    # Price-Volume interaction
    df_features['price_volume_interaction'] = df_features['y'] * df_features['volume_ratio_7']
    
    # Xóa NaN rows
    df_features = df_features.dropna()
    
    return df_features

# Tạo features
print("[INFO] Tạo lagged features (150 price lags + 30 volume lags + technical indicators)...")
df_features = create_lagged_features(df, lags=150)

# Chuẩn bị X (features) và y (target)
feature_cols = [col for col in df_features.columns if col != 'ds' and col != 'y' and col != 'volume']
X = df_features[feature_cols]
y = df_features['y']

print(f"[INFO] Số features: {len(feature_cols)}")
print(f"[INFO] Số samples: {len(X)}")

# Train CatBoost model
print("\n[INFO] Đang train CatBoost model với Volume features...")
model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='MAE',
    bootstrap_type='Bayesian',
    bagging_temperature=1.0,
    random_strength=1,
    l2_leaf_reg=3.0,
    verbose=50,
    random_state=42,
    task_type='CPU',
)

model.fit(X, y, verbose=False)
print("[INFO] ✅ Training hoàn tất!")

# Feature importance
feature_importance = model.get_feature_importance()
top_features_idx = np.argsort(feature_importance)[-15:]
print(f"\n[INFO] Top 15 features quan trọng nhất:")
for idx in top_features_idx[::-1]:
    print(f"  • {feature_cols[idx]}: {feature_importance[idx]:.4f}")

In [ ]:
# ========== DỰ ĐOÁN 90 NGÀY TIẾP THEO ==========
def forecast_catboost(model, df_features, df_original, periods=90):
    """
    Dự đoán n ngày tiếp theo sử dụng CatBoost (iterative forecasting)
    """
    df_temp = df_features.copy()
    forecasts = []
    
    # Lấy ngày cuối cùng trong data
    last_date = df_original['ds'].max()
    
    for step in range(periods):
        # Lấy row cuối cùng (features gần đây nhất)
        X_last = df_temp.iloc[[-1]][feature_cols]
        
        # Dự đoán giá trị tiếp theo
        yhat = model.predict(X_last)[0]
        forecasts.append(yhat)
        
        # Tạo row mới cho step tiếp theo
        new_row = X_last.copy()
        
        # Shift lagged features: lag_1 = lag_0, lag_2 = lag_1, ...
        for lag in range(150, 1, -1):
            if f'lag_{lag-1}' in new_row.columns:
                new_row[f'lag_{lag}'] = new_row[f'lag_{lag-1}'].values[0]
        
        # Lag_1 = giá dự đoán vừa rồi
        new_row['lag_1'] = yhat
        
        # Cập nhật moving averages
        last_y = df_temp.iloc[-1]['lag_1'] if 'lag_1' in df_temp.columns else df_original['y'].iloc[-1]
        new_row['ma7'] = np.mean([yhat] + [df_temp.iloc[-i]['lag_1'] for i in range(1, min(7, len(df_temp)))])
        new_row['ma30'] = np.mean([yhat] + [df_temp.iloc[-i]['lag_1'] for i in range(1, min(30, len(df_temp)))])
        
        df_temp = pd.concat([df_temp, new_row], ignore_index=True)
    
    # Tạo forecast dataframe
    forecast_dates = [last_date + timedelta(days=i) for i in range(1, periods + 1)]
    forecast_df = pd.DataFrame({
        'ds': forecast_dates,
        'yhat1': forecasts
    })
    
    return forecast_df

# Dự đoán 90 ngày
print("[INFO] Dự đoán 90 ngày tiếp theo...")
forecast_90d = forecast_catboost(model, df_features, df, periods=90)
print(f"[INFO] ✅ Dự đoán hoàn tất!")
print(f"\n📊 Dự đoán 90 ngày:")
print(forecast_90d.head(10))

In [ ]:
# ========== VẼ BIỂU ĐỒ FORECAST ==========
fig, ax = plt.subplots(figsize=(14, 6))

# Plot historical data
n_lags = 200
df_plot = df.tail(n_lags)
ax.plot(df_plot['ds'], df_plot['y'], label='Actual', marker='.', linewidth=2, color='black')

# Plot forecast
ax.plot(forecast_90d['ds'], forecast_90d['yhat1'], label='Forecast (90 days)', 
        marker='.', linewidth=2, color='red', linestyle='--')

ax.set_title('VN30 Lowest Price Forecast - CatBoost (Last 200 Days + 90 Days Forecast)', fontsize=12)
ax.set_xlabel('Date')
ax.set_ylabel('Price (VND)')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ========== TEST MAPE DIRECT 30 NGÀY ==========
def calculate_mape_direct_30days_catboost(model_catboost, df_original, df_features_full, test_days=30):
    """
    Tính MAPE cho dự đoán direct 30 ngày sử dụng CatBoost:
    - Train trên dữ liệu trước 30 ngày cuối
    - Dự đoán 1 lần cho 30 ngày tiếp theo
    - So sánh với thực tế
    """
    
    # Lấy ngày cuối cùng của dữ liệu
    last_date = df_original['ds'].max()
    train_end_date = last_date - pd.Timedelta(days=test_days)
    
    df_train = df_original[df_original['ds'] <= train_end_date].copy()
    df_test = df_original[df_original['ds'] > train_end_date].copy().reset_index(drop=True)
    
    print(f"[INFO] Train data: {len(df_train)} ngày (đến {train_end_date.date()})")
    print(f"[INFO] Test data: {len(df_test)} ngày (từ {(train_end_date + pd.Timedelta(days=1)).date()} đến {last_date.date()})")
    
    if len(df_test) < test_days:
        print(f"[WARNING] Chỉ có {len(df_test)} ngày test thay vì {test_days} ngày")
    
    # Train model trên dữ liệu train
    print("[INFO] Đang train CatBoost model trên dữ liệu train...")
    df_features_train = create_lagged_features(df_train, lags=150)
    
    X_train = df_features_train[feature_cols]
    y_train = df_features_train['y']
    
    model_test = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        loss_function='MAE',
        bootstrap_type='Bayesian',
        bagging_temperature=1.0,
        random_strength=1,
        l2_leaf_reg=3.0,
        verbose=False,
        random_state=42,
    )
    
    model_test.fit(X_train, y_train)
    
    # Dự đoán 1 lần cho test_days tiếp theo
    preds = []
    df_temp = df_features_train.copy()
    
    for step in range(test_days):
        X_last = df_temp.iloc[[-1]][feature_cols]
        yhat = model_test.predict(X_last)[0]
        preds.append(yhat)
        
        # Tạo row mới cho step tiếp theo
        new_row = X_last.copy()
        for lag in range(150, 1, -1):
            if f'lag_{lag-1}' in new_row.columns:
                new_row[f'lag_{lag}'] = new_row[f'lag_{lag-1}'].values[0]
        
        new_row['lag_1'] = yhat
        new_row['ma7'] = np.mean([yhat] + [df_temp.iloc[-i]['lag_1'] for i in range(1, min(7, len(df_temp)))])
        new_row['ma30'] = np.mean([yhat] + [df_temp.iloc[-i]['lag_1'] for i in range(1, min(30, len(df_temp)))])
        
        df_temp = pd.concat([df_temp, new_row], ignore_index=True)
    
    preds = np.array(preds)
    
    # Đảm bảo số lượng khớp với số ngày test
    n = min(len(preds), len(df_test))
    preds = preds[:n]
    test_dates = df_test['ds'][:n]
    test_y = df_test['y'][:n].values
    
    # So sánh với thực tế
    df_compare = pd.DataFrame({
        'ds': test_dates,
        'yhat': preds,
        'y': test_y
    })
    df_compare = df_compare.dropna(subset=['y', 'yhat'])
    
    if len(df_compare) == 0:
        print("[ERROR] Không có dữ liệu để so sánh MAPE")
        return None, None
    
    mape = mean_absolute_percentage_error(df_compare['y'], df_compare['yhat']) * 100
    mae = np.mean(np.abs(df_compare['y'] - df_compare['yhat']))
    rmse = np.sqrt(np.mean((df_compare['y'] - df_compare['yhat'])**2))
    
    print(f"\n[RESULT] MAPE Direct 30 ngày: {mape:.2f}%")
    print(f"[RESULT] MAE: {mae:.2f}")
    print(f"[RESULT] RMSE: {rmse:.2f}")
    print(f"[RESULT] Số ngày so sánh: {len(df_compare)}")
    print(f"[RESULT] Trung bình Actual: {df_compare['y'].mean():.2f}")
    print(f"[RESULT] Trung bình Predicted: {df_compare['yhat'].mean():.2f}")
    
    # Hiển thị sai số cao nhất
    df_compare['error'] = abs(df_compare['y'] - df_compare['yhat'])
    df_compare['error_pct'] = (df_compare['error'] / df_compare['y']) * 100
    
    print("\n[DETAIL] Top 5 ngày có sai số cao nhất:")
    top_errors = df_compare.nlargest(5, 'error_pct')[['ds', 'y', 'yhat', 'error', 'error_pct']]
    top_errors_display = top_errors.copy()
    top_errors_display['ds'] = pd.to_datetime(top_errors_display['ds']).dt.date
    print(top_errors_display.to_string(index=False))
    
    return mape, df_compare

# Chạy test MAPE
print("=" * 60)
print("TEST MAPE DIRECT 30 NGÀY (CATBOOST MODEL)")
print("=" * 60)
mape_score, comparison_df = calculate_mape_direct_30days_catboost(model, df, df_features, test_days=30)

In [ ]:
# ========== HIỂN THỊ KẾT QUẢ MAPE ==========
if 'mape_score' in locals() and mape_score is not None:
    print("=" * 60)
    print("KẾT QUẢ TEST MAPE DIRECT 30 NGÀY (CATBOOST)")
    print("=" * 60)
    print(f"MAPE: {mape_score:.2f}%")
    print(f"Số ngày test: {len(comparison_df)}")

    # Hiển thị thống kê chi tiết
    print("\nThống kê chi tiết:")
    print(comparison_df[['ds', 'y', 'yhat', 'error_pct']].describe())

    # Vẽ biểu đồ so sánh
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
    
    # Plot 1: So sánh giá trị
    ax1.plot(comparison_df['ds'], comparison_df['y'], label='Actual', marker='o', linewidth=2)
    ax1.plot(comparison_df['ds'], comparison_df['yhat'], label='Predicted', marker='x', linewidth=2)
    ax1.set_title(f'CatBoost: Actual vs Predicted (MAPE: {mape_score:.2f}%)')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('VN30 Low Price (VND)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Sai số theo ngày
    ax2.bar(range(len(comparison_df)), comparison_df['error_pct'])
    ax2.set_title('Error Percentage by Day')
    ax2.set_xlabel('Day')
    ax2.set_ylabel('Error %')
    ax2.grid(True, alpha=0.3)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Chưa có kết quả MAPE. Hãy chạy cell test MAPE trước.")

## 🎯 CatBoost Model - Ưu điểm vs Hạn chế

### ✅ Ưu điểm:
- **Xử lý non-linear patterns**: CatBoost giỏi bắt các mối quan hệ phi tuyến trong dữ liệu
- **Feature importance**: Cho biết feature nào quan trọng nhất cho dự đoán
- **Robust**: Ít bị overfitting nhất so với các tree-based models khác
- **Tốc độ**: Train nhanh hơn NeuralProphet

### ⚠️ Hạn chế:
- **Breakout prediction**: Vẫn khó dự đoán sudden breakout như NeuralProphet
- **Limited temporal context**: Dựa vào lagged features, không có temporal positional encoding
- **Memory usage**: Cần nhiều memory hơn cho features lớn

### 💡 Cải thiện:
- Ensemble CatBoost + ARIMA để xử lý trend và seasonality tốt hơn
- Thêm external features: VN-Index, trading volume, market sentiment
- Use residual modeling: Train CatBoost trên residuals của trend model

In [ ]:
# ========== PHÂN TÍCH FEATURE IMPORTANCE ==========
print("=" * 70)
print("📊 PHÂN TÍCH FEATURE IMPORTANCE - CATBOOST")
print("=" * 70)

# Lấy feature importance từ model
feature_importance = model.get_feature_importance()

# Sắp xếp và lấy top 15 features
top_n = 15
top_indices = np.argsort(feature_importance)[-top_n:][::-1]

print(f"\n🔝 Top {top_n} features quan trọng nhất:")
for i, idx in enumerate(top_indices, 1):
    feat_name = feature_cols[idx]
    importance = feature_importance[idx]
    print(f"  {i:2d}. {feat_name:20s} - Importance: {importance:8.4f}")

# Vẽ biểu đồ top 15 features
fig, ax = plt.subplots(figsize=(10, 6))
top_importance = feature_importance[top_indices]
top_names = [feature_cols[idx] for idx in top_indices]

ax.barh(range(len(top_names)), top_importance)
ax.set_yticks(range(len(top_names)))
ax.set_yticklabels(top_names)
ax.set_xlabel('Importance Score')
ax.set_title('CatBoost: Top 15 Feature Importance')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\n💡 Insights:")
print("  • lag_1 ... lag_7 thường là features quan trọng nhất (gần đây nhất)")
print("  • Moving averages (ma7, ma30) giúp bắt trend")
print("  • Volatility và ROC giúp detect momentum changes")

In [ ]:
# ========== PHÂN TÍCH NHANH CATBOOST MODEL PERFORMANCE ==========

print("=" * 70)
print("📊 PHÂN TÍCH NHANH PERFORMANCE CỦA CATBOOST")
print("=" * 70)

# Lấy 30 ngày cuối để phân tích
last_30 = df.tail(30)
y_last_30 = last_30['y'].values

trend_change = ((y_last_30[-1] / y_last_30[0]) - 1) * 100
trend_last_week = ((y_last_30[-1] / y_last_30[-7]) - 1) * 100 if len(y_last_30) >= 7 else 0

print(f"\n🔍 Phân tích 30 ngày gần nhất:")
print(f"  • Giá bắt đầu: {y_last_30[0]:.2f} VND")
print(f"  • Giá cuối: {y_last_30[-1]:.2f} VND")
print(f"  • Tăng trưởng toàn kỳ: {trend_change:.2f}%")
print(f"  • Trend tuần cuối: {trend_last_week:.2f}%")
print(f"  • Min 30d: {y_last_30.min():.2f}")
print(f"  • Max 30d: {y_last_30.max():.2f}")

# Kiểm tra nếu có kết quả MAPE
if 'mape_score' in locals() and mape_score is not None:
    print(f"\n✅ KẾT QUẢ MODEL TRỰC TIẾP:")
    print(f"  • MAPE: {mape_score:.2f}%")
    print(f"  • Số ngày test: {len(comparison_df)}")
    print(f"  • Trung bình Actual: {comparison_df['y'].mean():.2f} VND")
    print(f"  • Trung bình Predicted: {comparison_df['yhat'].mean():.2f} VND")
else:
    print(f"\n⚠️  Chưa chạy test MAPE. Hãy chạy cell test MAPE trước.")

print("\n💡 KẾT LUẬN:")
print(f"  ✓ CatBoost xử lý tốt non-linear patterns")
print(f"  ✓ Volume features giúp cải thiện độ chính xác")
print(f"  ✗ Vẫn khó bắt breakout đột biến ở cuối kỳ")

print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))
n_lags = 200
df_plot = df.tail(n_lags)

ax.plot(df_plot['ds'], df_plot['y'], label='Actual')
ax.plot(forecast_90d['ds'], forecast_90d['yhat1'], label='Forecast')

# Thử nhiều cách đặt tên bound, tự động bỏ qua nếu không có
lower_col = next((c for c in ['yhat1_lower','yhat_lower','lower','lo'] if c in forecast_90d.columns), None)
upper_col = next((c for c in ['yhat1_upper','yhat_upper','upper','hi'] if c in forecast_90d.columns), None)

if lower_col and upper_col:
    ax.fill_between(forecast_90d['ds'], forecast_90d[lower_col], forecast_90d[upper_col], alpha=0.25)

ax.set_title('VN30 Lowest Price Forecast (Last 200 Days)')
ax.set_xlabel('Date'); ax.set_ylabel('Price (VND)')
ax.legend(); ax.grid(True)
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

In [ ]:
# ========== DỰ ĐOÁN 7 NGÀY TIẾP THEO TỪ NGÀY HÔM NAY ==========
def forecast_next_n_days_catboost(model_catboost, df_features, df_original, periods=7):
    """
    Dự đoán n ngày tiếp theo (từ ngày hôm nay)
    """
    df_temp = df_features.copy()
    forecasts = []
    
    last_date = df_original['ds'].max()
    forecast_dates = [last_date + timedelta(days=i) for i in range(1, periods+1)]
    
    for step in range(periods):
        X_last = df_temp.iloc[[-1]][feature_cols]
        yhat = model_catboost.predict(X_last)[0]
        forecasts.append(yhat)
        
        # Update features cho step tiếp theo
        new_row = X_last.copy()
        for lag in range(150, 1, -1):
            if f'lag_{lag-1}' in new_row.columns:
                new_row[f'lag_{lag}'] = new_row[f'lag_{lag-1}'].values[0]
        
        new_row['lag_1'] = yhat
        new_row['ma7'] = np.mean([yhat] + [df_temp.iloc[-i]['lag_1'] for i in range(1, min(7, len(df_temp)))])
        new_row['ma30'] = np.mean([yhat] + [df_temp.iloc[-i]['lag_1'] for i in range(1, min(30, len(df_temp)))])
        
        df_temp = pd.concat([df_temp, new_row], ignore_index=True)
    
    return forecast_dates, forecasts

# Dự đoán 7 ngày tiếp theo
forecast_dates, forecast_values = forecast_next_n_days_catboost(model, df_features, df, periods=7)

print("=" * 60)
print("DỰ ĐOÁN VN30 LOW PRICE - 7 NGÀY TIẾP THEO (CATBOOST)")
print("=" * 60)
print(f"Model: CatBoost (1000 iterations, MAE loss)")
print(f"Training date: up to {df['ds'].max().date()}")
print(f"Forecast period: {forecast_dates[0].date()} to {forecast_dates[-1].date()}")
print("\n" + "=" * 60)

for date, value in zip(forecast_dates, forecast_values):
    print(f"📅 {date.date()} | 💰 {value:.2f} VND")

print("=" * 60)